# 02 · Pipeline de la entrega 2

Corre el pipeline completo de la entrega 2 y muestra los resultados: Landing → Bronze → Silver → streaming → Gold. En Colab va sin serving, porque no hay Cassandra; la parte de Cassandra se corre en local (README, Quickstart).

Colab ya trae PySpark 4.0.4 y Java 21. Tarda unos 6 minutos con las 2 CPU de Colab.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/cloud-provider-analytics")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/gonrc/cloud-provider-analytics.git", str(repo)], check=True)
    os.chdir(repo)
elif Path.cwd().name == "notebooks":
    os.chdir("..")
print("raíz del repo:", Path.cwd())

## 1. Correr el pipeline

`--sin-serving` corre todo menos la carga en Cassandra. La última línea tiene que decir `controles fallidos: ninguno`.

In [ ]:
!PYTHONPATH=src python -m cpa.pipeline --sin-serving 2>&1 | grep -v -E "WARN|log4j|setLogLevel|incubator|NativeCodeLoader"

## 2. Conteos y controles de la corrida

Cada paso deja su registro en `evidence/runs/`. Por tabla, lo que entra es igual a lo que sale más lo que va a quarantine.

In [ ]:
import json, pandas as pd
log = json.loads(sorted(Path("evidence/runs").glob("*_pipeline.json"))[-1].read_text())
print("run_id:", log["run_id"])
display(pd.DataFrame([{"control": k, "ok": v["ok"], "detalle": v["detail"]} for k, v in log["checks"].items()]))
pd.DataFrame([{"tabla": k, **{c: v for c, v in d.items() if not isinstance(v, dict)}} for k, d in log["counts"].items()])

## 3. Streaming: un registro por micro-lote

In [ ]:
pd.read_csv("evidence/02_entrega/streaming_lotes.csv")

## 4. Quarantine y Gold

In [ ]:
q = pd.read_parquet("datalake/quarantine/usage_events")
display(q.groupby(["dq_rule", "dq_reason"]).size().rename("filas").reset_index())
g = pd.read_parquet("datalake/gold/finops/org_daily_usage_by_service")
print("filas en Gold:", len(g), "| costo total:", round(g.daily_cost_usd.sum(), 2), "USD")
g.sort_values(["org_id", "usage_date", "service"]).head()

In [ ]:
a = pd.read_parquet("datalake/gold/finops/cost_anomaly_mart")
print("días marcados como anomalía:", int(a.is_anomaly.sum()), "de", len(a))
a[a.is_anomaly].sort_values("anomaly_score", ascending=False)[["org_id", "usage_date", "service", "daily_cost_usd", "baseline_daily_cost_usd", "anomaly_score"]].head(10)